In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

#Plot Styling
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12,5)
plt.rcParams['fontsize'] = 12



In [4]:
# Import Clean Data
df = pd.read_csv(r'C:\Users\chapa\Desktop\retail-sales-analytics\data\clean\sales_clean.csv')
df.head(5)

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Sales,Quantity,Discount,Profit,Order Year,Order Month,Order Month Name,Order Quarter,Days to Ship,Profit Margin Percentage
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,261.9600,2,0.00,41.9136,2016,11,November,4,3,16.00
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,731.9400,3,0.00,219.5820,2016,11,November,4,3,30.00
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,14.6200,2,0.00,6.8714,2016,6,June,2,4,47.00
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,957.5775,5,0.45,-383.0310,2015,10,October,4,7,-40.00
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,22.3680,2,0.20,2.5164,2015,10,October,4,7,11.25


## 1. Revenue & Profitability

In [7]:
total_revenue = df['Sales'].sum()
total_profit = df['Profit'].sum()
profit_margin = (total_profit / total_revenue) * 100
total_orders = df['Order ID'].nunique()
total_customers = df['Customer ID'].nunique()

print("=" * 50)
print("       INSIGHT 1: Business Health")
print("=" * 50)
print(f"  Total Revenue   : ${total_revenue:,.2f}")
print(f"  Total Profit   : ${total_profit:,.2f}")
print(f"  Profit Margin   : {profit_margin:,.2f}%")
print(f"  Total Orders   : {total_orders:,}")
print(f"  Total Customers   : {total_customers:,}")
print("=" * 50)

       INSIGHT 1: Business Health
  Total Revenue   : $2,297,200.86
  Total Profit   : $286,397.02
  Profit Margin   : 12.47%
  Total Orders   : 5,009
  Total Customers   : 793


In [8]:
# Year over year revenue growth
yoy = df.groupby('Order Year')['Sales'].sum()
print("\nYear Over Year Revenue")
for year, sales in yoy.items():
    print(f"  {year} : ${sales:,.2f}")


Year Over Year Revenue
  2014 : $484,247.50
  2015 : $470,532.51
  2016 : $609,205.60
  2017 : $733,215.26


In [11]:
# Growth Rate
years = yoy.index.tolist()
for i in range(1, len(years)):
    growth = ((yoy[years[i]] - yoy[years[i-1]]) / yoy[years[i-1]]) * 100
    print(f"  {years[i-1]} -> {years[i]} Growth: {growth:.1f}%")

  2014 -> 2015 Growth: -2.8%
  2015 -> 2016 Growth: 29.5%
  2016 -> 2017 Growth: 20.4%


## 2. Category & Segment Insights

In [16]:
# Category Performance Insights
category_perf = df.groupby('Category').agg(
    Total_Sales = ('Sales', 'sum'),
    Total_Profit = ('Profit','sum'),
    Total_Orders = ('Order ID', 'count'),
    Avg_Discount = ('Discount', 'mean')
    ).round(2)

category_perf['Profit Margin %'] = (
    (category_perf['Total_Profit'] / category_perf['Total_Sales']) * 100
    ).round(1)

category_perf = category_perf.sort_values('Total_Sales', ascending=False)

print("=" * 50)
print("      INSIGHT 2: CATEGORY PERFORMANCE")
print("=" * 50)
print(category_perf.to_string())


      INSIGHT 2: CATEGORY PERFORMANCE
                 Total_Sales  Total_Profit  Total_Orders  Avg_Discount  Profit Margin %
Category                                                                               
Technology         836154.03     145454.95          1847          0.13             17.4
Furniture          741999.80      18451.27          2121          0.17              2.5
Office Supplies    719047.03     122490.80          6026          0.16             17.0


In [17]:
# Segment Performance
segment_perf = df.groupby('Segment').agg(
    Total_Sales = ('Sales', 'sum'),
    Total_Profit = ('Profit', 'sum'),
    Avg_Order = ('Sales', 'mean')
).round(2)

segment_perf['Profit Margin %'] = (
    (segment_perf['Total_Profit'] / segment_perf['Total_Sales']) * 100
).round(1)

print("\n" + "=" * 50)
print("      INSIGHT 3: SEGMENT PERFORMANCE")
print("=" * 50)
print(segment_perf.to_string())


      INSIGHT 3: SEGMENT PERFORMANCE
             Total_Sales  Total_Profit  Avg_Order  Profit Margin %
Segment                                                           
Consumer      1161401.34     134119.21     223.73             11.5
Corporate      706146.37      91979.13     233.82             13.0
Home Office    429653.15      60298.68     240.97             14.0


## 3. Discout & Shipping Insights

In [20]:
# Discount Impact
df['Discount Bucket'] = pd.cut(df['Discount'],
                              bins=[0, 0.1, 0.2, 0.3,0.4,0.5,1.0],
                              labels=['0-10%', '10-20%', '20-30%',
                                      '30-40%', '40-50%', '50%+'],
                              include_lowest=True)

discount_impact = df.groupby('Discount Bucket', observed=True).agg(
    Avg_Profit = ('Profit', 'mean'),
    Total_Orders = ('Order ID', 'count')
).round(2)

print("=" * 50)
print("      INSIGHT 4: DISCOUNT IMPACT")
print("=" * 50)
print(discount_impact.to_string())

      INSIGHT 4: DISCOUNT IMPACT
                 Avg_Profit  Total_Orders
Discount Bucket                          
0-10%                 67.46          4892
10-20%                24.74          3709
20-30%               -45.68           227
30-40%              -109.22           233
40-50%              -298.70            77
50%+                 -89.44           856


In [21]:
# We have to find a threshold where we break even with discounts
losing = discount_impact[discount_impact['Avg_Profit'] < 0]
if not losing.empty:
    threshold = losing.index[0]
    print(f"\n Profit turns negative at discount bucket: {threshold}")


 Profit turns negative at discount bucket: 20-30%


In [22]:
# Lets dig into shipping
ship_perf = df.groupby('Ship Mode').agg(
    Avg_Days = ('Days to Ship', 'mean'),
    Total_Orders = ('Order ID', 'count'),
    Total_Sales = ('Sales', 'sum')
).round(2)

ship_perf = ship_perf.sort_values('Avg_Days')

print("\n" + "=" * 50)
print("      INSIGHT 5: SHIPPING PERFORMANCE")
print("=" * 50)
print(ship_perf.to_string())


      INSIGHT 5: SHIPPING PERFORMANCE
                Avg_Days  Total_Orders  Total_Sales
Ship Mode                                          
Same Day            0.04           543    128363.12
First Class         2.18          1538    351428.42
Second Class        3.24          1945    459193.57
Standard Class      5.01          5968   1358215.74


## 4. Top & Bottom Performers

In [23]:
# Top and Bottom Product Performance
top_products = df.groupby('Product Name').agg(
    Total_Sales = ('Sales', 'sum'),
    Total_Profit = ('Profit', 'sum')
).round(2).sort_values('Total_Sales', ascending=False)

print("=" * 50)
print("      INSIGHT 6: TOP 5 PRODUCTS BY SALES")
print("=" * 50)
print(top_products.head().to_string())

print("\n" + "=" * 50)
print("      INSIGHT 6: BOTTOM 5 PRODUCTS BY PROFIT")
print("=" * 50)
print(top_products.sort_values('Total_Profit').head().to_string())


      INSIGHT 6: TOP 5 PRODUCTS BY SALES
                                                                             Total_Sales  Total_Profit
Product Name                                                                                          
Canon imageCLASS 2200 Advanced Copier                                           61599.82      25199.93
Fellowes PB500 Electric Punch Plastic Comb Binding Machine with Manual Bind     27453.38       7753.04
Cisco TelePresence System EX90 Videoconferencing Unit                           22638.48      -1811.08
HON 5400 Series Task Chairs for Big and Tall                                    21870.58          0.00
GBC DocuBind TL300 Electric Binding System                                      19823.48       2233.51

      INSIGHT 6: BOTTOM 5 PRODUCTS BY PROFIT
                                                          Total_Sales  Total_Profit
Product Name                                                                       
Cubify CubeX 3D Printer 

In [25]:
# Lets get Regional Performance Info
region_perf = df.groupby('Region').agg(
    Total_Sales = ('Sales','sum'),
    Total_Profit = ('Profit', 'sum'),
    Total_Orders = ('Order ID', 'count')
).round(2)

region_perf['Profit Margin %'] = (
    (region_perf['Total_Profit'] / region_perf['Total_Sales']) * 100
).round(1)

region_perf = region_perf.sort_values('Total_Sales', ascending=False)

print("\n" + "=" * 50)
print("      INSIGHT 7: REGIONAL PERFORMANCE")
print("=" * 50)
print(region_perf.to_string())


      INSIGHT 7: REGIONAL PERFORMANCE
         Total_Sales  Total_Profit  Total_Orders  Profit Margin %
Region                                                           
West       725457.82     108418.45          3203             14.9
East       678781.24      91522.78          2848             13.5
Central    501239.89      39706.36          2323              7.9
South      391721.90      46749.43          1620             11.9


## 5. Export to Excel

In [29]:
# Export our insight to excel

with pd.ExcelWriter(r'C:\Users\chapa\Desktop\retail-sales-analytics\data\clean\sales_insights_report.xlsx',
                   engine='openpyxl') as writer:
    
    # Sheet 1
    df.to_excel(writer, sheet_name='Clean Data', index=False)
    
    # Sheet 2
    category_perf.to_excel(writer, sheet_name='Category Performance')
    
    #Sheet 3
    segment_perf.to_excel(writer, sheet_name='Segment Performance')
    
    # Sheet 4
    discount_impact.to_excel(writer, sheet_name='Discount Analysis')
    
    #Sheet 5
    ship_perf.to_excel(writer, sheet_name='Shipping Performance')
    
    #Sheet 6
    region_perf.to_excel(writer, sheet_name='Regional Performance')
    
    #Sheet 7
    top_products.head(20).to_excel(writer, sheet_name='Top 20 Products')
    
print(" Insights report exported to data/clean/sales_insights_report.xlsx")
print("   Sheets created:")
print("   - Clean Data")
print("   - Category Performance")
print("   - Segment Performance")
print("   - Discount Analysis")
print("   - Shipping Performance")
print("   - Regional Performance")
print("   - Top 20 Products")

 Insights report exported to data/clean/sales_insights_report.xlsx
   Sheets created:
   - Clean Data
   - Category Performance
   - Segment Performance
   - Discount Analysis
   - Shipping Performance
   - Regional Performance
   - Top 20 Products


# 6. Key Business Insights & Recommendations

## Insight 1: Revenue Growth
- Revenue has grown year over year showing consistent business expansion
- Identify peak months to align inventory and marketing accordingly

## Insight 2: Category Performance
- Technology Drives the highest revenue but examine profit margins closely
- Furniture may show high sales but lower profitability due to discounts
- Office Supplies show the most consistent profit margins

## Insight 3: Discount Strategy is Hurting Profit
- Orders with discount above 20% consitently generate negative profit
- **Recommendation:** Cap discounts at 20% across all categories
- High discount rates in Furniture are the biggest profit drain

## Insight 4: Regional Opportunities
- The west and east regions lead in both sales and profit
- The central region underperforms relative to its order volume
- **Recommendation:** Review Central region pricing and product mix

## Insight 5: Shipping Efficiency
- Standard Class is the most used shipping method
- Same Day Shipping is underutilized but could be a premium upsell opportunity 
- **Recommendation:** Promote faster shipping options to high value customers

## Overall Recommendation
Focus on three levers to improve profitability:
1. **Cap discounts at 20%** to protect margins
2. **Invest in the West & East regions** where ROI is highest
3. **Review Furniture category pricing** to improve its profit margin